In [1]:
import numpy as np
import pandas as pd
import os

input_dir = '/kaggle/input/playground-series-s5e10'

In [2]:
from sklearn.preprocessing import StandardScaler 
from sklearn.decomposition import PCA

def ingest(input_dir):
    target = 'accident_risk'

    train_csv = pd.read_csv(os.path.join(input_dir, 'train.csv'))
    X_train_df = train_csv.drop(columns=['id', target])
    X_test_df = pd.read_csv(os.path.join(input_dir, 'test.csv')).drop(columns=['id'])
    # print(X_train_df.dtypes)
    features = pd.concat([X_train_df, X_test_df])
    
    num_cols = features.select_dtypes(include=['float64', 'int64']).columns

    features[num_cols] = features[num_cols].fillna(features[num_cols].mean())
    
    features = pd.get_dummies(features)
    
    y_train = train_csv[target]
    
    X_train = features.iloc[:len(y_train)].copy()
    X_test = features.iloc[len(y_train):].copy()

    scaler = StandardScaler()
    
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])  

    X_train = pd.DataFrame(X_train, columns=X_train.columns)
    X_test = pd.DataFrame(X_test, columns=X_test.columns)
    
    features = pd.DataFrame(pd.concat([X_train, X_test]), columns=X_train.columns)
    
    bool_cols = features.select_dtypes(include='bool').columns
    features[bool_cols] = features[bool_cols].astype(int)
    
    features[num_cols] = features[num_cols].astype('float32')
    # print(features)

    X_train = features.iloc[:len(y_train)]
    X_test = features.iloc[len(y_train):]

    pca = PCA(n_components=18)

    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)
    
    return X_train, y_train, X_test

X_train_df, y_train_df, X_test_df = ingest(input_dir)
print(type(X_test_df))
print(X_train_df.shape, y_train_df.shape, X_test_df.shape)
X_train_df

<class 'numpy.ndarray'>
(517754, 18) (517754,) (172585, 18)


array([[-1.37378803e+00, -4.44269461e-01,  1.53059293e-01, ...,
         1.72652771e-01, -2.72260446e-15,  1.23080291e-15],
       [ 2.17739731e-01,  2.33621352e-01,  1.24324172e+00, ...,
        -1.69060124e-01,  2.04230143e-15,  2.79543533e-15],
       [ 1.25233668e+00,  1.91691211e+00, -2.89558944e-01, ...,
        -7.33448865e-01, -1.21263829e-15,  5.82682500e-15],
       ...,
       [-8.44884849e-01,  1.23860263e-01,  1.68301095e+00, ...,
         7.17188795e-01, -1.29276056e-16,  5.29097651e-17],
       [ 1.53150548e+00, -3.98553522e-01,  1.64820300e+00, ...,
        -1.94083132e-01,  7.95872836e-17,  1.60045048e-17],
       [ 1.50347945e-01, -2.08927841e-01, -1.09197183e-01, ...,
        -5.34608453e-01, -2.05365202e-16,  3.05933545e-17]])

In [3]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train_set, X_test_set, y_train_set, y_test_set = train_test_split(X_train_df, y_train_df, test_size=0.2, random_state=42)

# X_train = torch.tensor(X_train_set.values, dtype=torch.float32)
# X_test = torch.tensor(X_test_set.values, dtype=torch.float32)
# y_train = torch.tensor(y_train_set.values, dtype=torch.float32).unsqueeze(1)
# y_test = torch.tensor(y_test_set.values, dtype=torch.float32).unsqueeze(1)

X_train = torch.from_numpy(X_train_set.astype('float32'))
X_test = torch.from_numpy(X_test_set.astype('float32'))
y_train = torch.from_numpy(y_train_set.to_numpy().astype('float32')).unsqueeze(1)
y_test = torch.from_numpy(y_test_set.to_numpy().astype('float32')).unsqueeze(1)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

batch_size = 64

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [4]:
from torch import nn

class CustomMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(X_train.shape[1], 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

net = CustomMLP()

In [5]:
def train(net, dataloader, num_epochs, lr, m, device):
    torch.manual_seed(42)
    net = net.to(device)
    loss = nn.MSELoss()
    optim = torch.optim.SGD(net.parameters(), lr=lr, momentum=m)
    for epoch in range(num_epochs):
        net.train()
        running, seen = 0.0, 0
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            optim.zero_grad()
            pred = net(X)
            l = loss(pred, y)
            l.backward()
            optim.step()
            running += l.item() * X.size(0)
            seen += X.size(0)
            mse = running / seen
            rmse = mse ** 0.5

        print(f'Epoch: {epoch + 1} RMSE Loss: {rmse:.4f}')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
train(net, train_loader, 10, 0.1, 0.9, device)

Using: cuda
Epoch: 1 RMSE Loss: 0.0699
Epoch: 2 RMSE Loss: 0.0629
Epoch: 3 RMSE Loss: 0.0619
Epoch: 4 RMSE Loss: 0.0615
Epoch: 5 RMSE Loss: 0.0611
Epoch: 6 RMSE Loss: 0.0611
Epoch: 7 RMSE Loss: 0.0609
Epoch: 8 RMSE Loss: 0.0609
Epoch: 9 RMSE Loss: 0.0608
Epoch: 10 RMSE Loss: 0.0606


In [6]:
def eval_rmse(net, dataloader, device):
    net.eval()
    squared_err, total = 0.0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = net(X)
            squared_err += ((pred - y) ** 2).sum().item()
            total += X.size(0)

    rmse = (squared_err / total) ** 0.5
    return rmse

eval_rmse = eval_rmse(net, test_loader, device)
print(f'Eval RMSE: {eval_rmse:.4f}')

Eval RMSE: 0.0583


In [7]:
# X_test_tensor = torch.tensor(X_test_df.values, dtype=torch.float32).to(device)
X_test_tensor = torch.from_numpy(X_test_df.astype('float32')).to(device)
net.eval()

with torch.no_grad():
    preds = net(X_test_tensor).squeeze(1).cpu().numpy()

ids = pd.read_csv(os.path.join(input_dir, 'test.csv'))['id']

submission = pd.DataFrame({
    'id': ids,
    'accident_risk': preds
})
submission.to_csv('submission.csv', index=False)